TODO:

Choose just one caption for each image, or keep them all
Field boosting is to be performed in the hybrid-search part

For the cross modal search, the image embedding has to saved once, so the captions should either be collected in the same document with their respective embeddings or only one caption is kept obtaining effectively only one caption per image and vice versa.

## Text-based Search

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENSEARCH_USER = os.getenv("OPENSEARCH_USER")
OPENSEARCH_PASSWORD = os.getenv("OPENSEARCH_PASSWORD")
OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST")
OPENSEARCH_PORT = os.getenv("OPENSEARCH_PORT")

In [2]:
import pprint as pp
from opensearchpy import OpenSearch
from opensearchpy import helpers

index_name = OPENSEARCH_USER + '_project'
# Create the client with SSL/TLS enabled, but hostname verification disabled.
client = OpenSearch(
    hosts = [{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_compress = True, # enables gzip compression for request bodies
    http_auth = (OPENSEARCH_USER, OPENSEARCH_PASSWORD),
    use_ssl = True,
    url_prefix = 'opensearch_v3',
    verify_certs = False,
    ssl_assert_hostname = False,
    ssl_show_warn = False
)

if client.indices.exists(index=index_name):

    resp = client.indices.open(index=index_name)
    print(resp)

    print('\n----------------------------------------------------------------------------------- INDEX SETTINGS')
    settings = client.indices.get_settings(index=index_name)
    pp.pprint(settings)

    print('\n----------------------------------------------------------------------------------- INDEX MAPPINGS')
    mappings = client.indices.get_mapping(index=index_name)
    pp.pprint(mappings)

    print('\n----------------------------------------------------------------------------------- INDEX #DOCs')
    print(client.count(index=index_name))
else:
    print("Index does not exist.")


{'acknowledged': True, 'shards_acknowledged': True}

----------------------------------------------------------------------------------- INDEX SETTINGS
{'uservl07_project': {'settings': {'index': {'creation_date': '1774886704756',
                                             'knn': 'true',
                                             'knn.derived_source': {'enabled': 'true'},
                                             'number_of_replicas': '0',
                                             'number_of_shards': '4',
                                             'provided_name': 'uservl07_project',
                                             'refresh_interval': '1s',
                                             'replication': {'type': 'DOCUMENT'},
                                             'similarity': {'bm25-0-75': {'b': '0.75',
                                                                          'k1': '0.0',
                                                                      

In [3]:
from datasets import load_dataset
import aiohttp

ds = load_dataset("HuggingFaceM4/COCO", split="validation",
                  storage_options={'client_kwargs': {'timeout': aiohttp.ClientTimeout(total=36000)}}
                  )
print(ds.column_names)
pp.pprint(ds[0])
print(ds[0]['sentences']['raw'])
print(ds[0]['image'].width)


['image', 'filepath', 'sentids', 'filename', 'imgid', 'split', 'sentences', 'cocoid']
{'cocoid': 184613,
 'filename': 'COCO_val2014_000000184613.jpg',
 'filepath': 'COCO_val2014_000000184613.jpg',
 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=500x336 at 0x140D0EED0>,
 'imgid': 2,
 'sentences': {'imgid': 2,
               'raw': 'A child holding a flowered umbrella and petting a yak.',
               'sentid': 474921,
               'tokens': ['a',
                          'child',
                          'holding',
                          'a',
                          'flowered',
                          'umbrella',
                          'and',
                          'petting',
                          'a',
                          'yak']},
 'sentids': [474921, 479322, 479334, 481560, 483594],
 'split': 'val'}
A child holding a flowered umbrella and petting a yak.
500


Set up an OpenSearch index and define mappings for image identifiers,
captions, and COCO metadata (categories, image dimensions).

In [4]:
#Delete index if necessary

pp.pprint(client.indices.delete(index=index_name))

{'acknowledged': True}


In [5]:
index_body = {
   "settings":{
      "index":{
         "number_of_replicas":0,
         "number_of_shards":4,
         "refresh_interval":"-1",
         "knn":"true"
      },
      "similarity":{
          "bm25-20-75":{
              "type":"BM25",
              "k1":2.0,
              "b":0.75
          },
          "bm25-12-0":{
              "type":"BM25",
              "k1":1.2,
              "b":0.0
          },
          "bm25-0-75":{
              "type":"BM25",
              "k1":0.0,
              "b":0.75
          }

      }
   },
   "mappings":{
       "dynamic":      "strict",
       "properties":{
            #"_id": { "type": "keyword"},
            "image_id": { "type": "keyword"},
            "caption": { 
                "type": "text",
                "analyzer": "standard",
                "similarity":"BM25",
                "fields":{
                    "bm25_20_75": {
                        "type": "text",
                        "analyzer": "standard",
                        "similarity":"bm25-20-75"
                    },
                    "bm25_12_0": {
                        "type": "text",
                        "analyzer": "standard",
                        "similarity":"bm25-12-0"
                    },
                    "bm25_0_75": {
                        "type": "text",
                        "analyzer": "standard",
                        "similarity":"bm25-0-75"
                     }
                  },
            },
            "categories": { "type": "keyword"},
            "width": { "type": "integer"},
            "height": { "type": "integer"},

            "SBERT_embedding": {
                "type":"knn_vector",
                "dimension": 768,
                "method":{
                    "name":"hnsw",
                    "space_type":"innerproduct",
                    "engine":"faiss",
                    "parameters":{
                        "ef_construction":256,
                        "m":48
                    }
                },
            },

            "BGE_embedding": {
                "type":"knn_vector",
                "dimension": 384,
                "method":{
                    "name":"hnsw",
                    "space_type":"innerproduct",
                    "engine":"faiss",
                    "parameters":{
                        "ef_construction":256,
                        "m":48
                    }
                }
            }
      }
   }
}

if client.indices.exists(index=index_name):
    print("Index already existed. Nothing to be done.")
else:        
    response = client.indices.create(index=index_name, body=index_body)
    print('\nCreating index:')
    print(response)



Creating index:
{'acknowledged': True, 'shards_acknowledged': True, 'index': 'uservl07_project'}


Load dataset captions in index

In [ ]:
def index_data(client, index_name, dataset):
    actions= []
    inserted_imgids = set()
    for i,item in enumerate(dataset):
        if item['sentences']['imgid'] in inserted_imgids:
            continue
        action={
            "_id":item['sentences']['sentid'],
            "image_id":item['sentences']['imgid'],
            "caption":item['sentences']['raw'],
            #"categories":
            "width":item['image'].width,
            "height":item['image'].height
        }
        actions.append(action)
        inserted_imgids.add(item['sentences']['imgid'])
        if len(actions) == 1000:
            helpers.bulk(client, actions, index=index_name)
            print(f"Indexed {i+1} documents")
            actions = []
    if actions:
        helpers.bulk(client, actions, index=index_name)
        print(f"Indexed {len(dataset)} documents in total")
count = client.count(index=index_name)
print(count['count'])

if count['count'] == 0:
    index_data(client, index_name, ds)
else:
    print("Index already contains documents. Nothing to be done.")

0
Indexed 4998 documents
Indexed 9999 documents
Indexed 15002 documents
Indexed 20006 documents
Indexed 25006 documents


In [7]:
client.indices.put_settings(index=index_name, body={"index": {"refresh_interval": "1s"}}) #set refresh interval to 1s to make all operations available for search after 1s
client.indices.refresh(index=index_name) #refresh immediately

{'_shards': {'total': 4, 'successful': 4, 'failed': 0}}

In [8]:

count = client.count(index=index_name)
print(count['count'])

# check a random document
result = client.search(
    index=index_name,
    body={"query": {"match_all": {}}, "size": 1}
)
print(result['hits']['hits'][0]['_source'])

5000
{'image_id': 2, 'caption': 'A child holding a flowered umbrella and petting a yak.', 'width': 500, 'height': 336}


Perform the same query computing similarity with BM25 with different parameters.
For field boosting, idea is to use another field, but the dataset doesn't have a categories objects

In [9]:
query_txt = ds[0]['sentences']['raw']


bm25_fields = ["caption", "caption.bm25_20_75", "caption.bm25_12_0", "caption.bm25_0_75"]

for field in bm25_fields:
    query ={
        "size": 10,
        "query": {
            "match": {
                field: {
                    "query": query_txt
                }
            }
        }
    }
    response = client.search(body=query, index=index_name)
    #pp.pprint(response)
    print(f'\nSearch results for field "{field}":')
    for hit in response['hits']['hits']:
        print(f"Score: {hit['_score']:3.4f}, Doc ID: {hit['_id']}, Image ID: {hit['_source']['image_id']}, Caption: {hit['_source']['caption']}")


Search results for field "caption":
Score: 15.1620, Doc ID: 474921, Image ID: 2, Caption: A child holding a flowered umbrella and petting a yak.
Score: 5.4905, Doc ID: 192463, Image ID: 33253, Caption: a close up of a child holding a closed umbrella
Score: 4.6502, Doc ID: 402111, Image ID: 7647, Caption: a woman and a child holding umbrellas and smiling
Score: 4.3947, Doc ID: 373329, Image ID: 8726, Caption: A child, wearing a cat costume and umbrella, stands before a brick building.
Score: 4.2200, Doc ID: 403240, Image ID: 31747, Caption: A man in shirt and tie holding an umbrella.
Score: 3.9660, Doc ID: 184436, Image ID: 25158, Caption: A young child is holding a blue skateboard.
Score: 3.9541, Doc ID: 488669, Image ID: 6873, Caption: An adult standing holding surfboard and a child kneeling by water.
Score: 3.8189, Doc ID: 207678, Image ID: 27267, Caption: A woman holding a child on a skateboard.
Score: 3.7955, Doc ID: 717298, Image ID: 22998, Caption: A man holding an umbrella in a

## Semantic Text Search

TODO: upload so to encode only the inserted images

In [15]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd


df = ds.to_pandas()
#raw_captions = df['sentences'].apply(lambda x: x['raw']).head()

df = pd.DataFrame(
    {
        "sentid": df['sentences'].apply(lambda x: x['sentid']),
        "imgid": df['sentences'].apply(lambda x: x['imgid']),
        "raw": df['sentences'].apply(lambda x: x['raw']),
    }
).drop_duplicates(subset=['imgid'], keep='first').reset_index(drop=True)
df.head()

sbert_model = SentenceTransformer('all-mpnet-base-v2')
bge_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
sbert_encoding = sbert_model.encode(df['raw'], show_progress_bar=True)
bge_encoding = bge_model.encode(df['raw'], show_progress_bar=True)

print("SBERT Encodings shape:", sbert_encoding.shape)
print("BGE Encodings shape:", bge_encoding.shape)

df['sbert_embedding'] = list(sbert_encoding)
df['bge_embedding'] = list(bge_encoding)

df.head()
df.to_parquet('coco_captions_embeddings.parquet', index=False)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

SBERT Encodings shape: (5000, 768)
BGE Encodings shape: (5000, 384)


In [16]:
df.head()

,sentid,imgid,raw,sbert_embedding,bge_embedding
0,474921,2,A child holding a flowered umbrella and pettin...,"[0.043276522, -0.07047357, 0.0053688735, 0.023...","[0.011913764, -0.0007062534, 0.08118155, -0.02..."
1,124166,15,A narrow kitchen filled with appliances and co...,"[-0.010408675, -0.054555293, -0.01693293, -0.0...","[0.016339002, 0.012613272, 0.043380924, -0.013..."
2,96821,36,A little girl holding a kitten next to a blue ...,"[0.028164648, 0.0203255, 0.018007534, -0.01604...","[0.04609613, 0.012474204, 0.047648523, 0.02186..."
3,204010,53,A toilet sitting in a bathroom next to a sink.,"[0.01762949, -0.0095861005, -0.035485506, -0.0...","[0.05711122, -0.045034394, 0.067034, -0.071054..."
4,444042,57,There are two sinks next to two mirrors.,"[0.046684165, -0.025488446, -0.02733047, -0.00...","[0.021507654, 0.023089329, 0.057892725, -0.022..."


## Update index


In [17]:
df = pd.read_parquet('coco_captions_embeddings.parquet')
df.head()

,sentid,imgid,raw,sbert_embedding,bge_embedding
0,474921,2,A child holding a flowered umbrella and pettin...,"[0.043276522, -0.07047357, 0.0053688735, 0.023...","[0.011913764, -0.0007062534, 0.08118155, -0.02..."
1,124166,15,A narrow kitchen filled with appliances and co...,"[-0.010408675, -0.054555293, -0.01693293, -0.0...","[0.016339002, 0.012613272, 0.043380924, -0.013..."
2,96821,36,A little girl holding a kitten next to a blue ...,"[0.028164648, 0.0203255, 0.018007534, -0.01604...","[0.04609613, 0.012474204, 0.047648523, 0.02186..."
3,204010,53,A toilet sitting in a bathroom next to a sink.,"[0.01762949, -0.0095861005, -0.035485506, -0.0...","[0.05711122, -0.045034394, 0.067034, -0.071054..."
4,444042,57,There are two sinks next to two mirrors.,"[0.046684165, -0.025488446, -0.02733047, -0.00...","[0.021507654, 0.023089329, 0.057892725, -0.022..."


In [18]:
def update_index(client, index_name, df):
    actions = []
    for i, row in df.iterrows():
        action = {
            "_op_type": "update",
            "_index": index_name,
            "_id": row['sentid'],
            "doc": {
                "SBERT_embedding": row['sbert_embedding'],
                "BGE_embedding": row['bge_embedding']
            }
        }
        actions.append(action)
        if len(actions) == 1000:
            helpers.bulk(client, actions)
            print(f"Updated {i+1} documents")
            actions = []
    if actions:
        helpers.bulk(client, actions)
        print(f"Updated {len(df)} documents in total")

In [19]:

update_index(client, index_name, df)

Updated 1000 documents
Updated 2000 documents
Updated 3000 documents
Updated 4000 documents
Updated 5000 documents


In [20]:
#query a random document to check if the update was successful
result = client.search(
    index=index_name,
    body={"query": {"match_all": {}}, "size": 1}
)
pp.pprint(result['hits']['hits'][0]['_source'])

{'BGE_embedding': [0.03406848,
                   0.03607354,
                   0.083689466,
                   -0.009539747,
                   -0.006600081,
                   -0.0013830524,
                   0.07237252,
                   0.033579133,
                   0.097500905,
                   0.018711755,
                   -0.035702437,
                   -0.020488936,
                   0.053530943,
                   0.066078424,
                   0.0044054026,
                   -0.008169162,
                   -0.009327146,
                   0.053750288,
                   -0.08079825,
                   -0.01815123,
                   0.14487408,
                   -0.04196044,
                   -0.003295689,
                   -0.051035877,
                   -0.025120702,
                   -0.019775096,
                   0.0059935725,
                   -0.015238627,
                   -0.017460197,
                   -0.09413924,
                   -0.042522

In [ ]:
client.indices.close(index=index_name)

{'acknowledged': True,
 'shards_acknowledged': True,
 'indices': {'uservl07_project': {'closed': True}}}